# Long-tailed video understanding on UCF-101

Frozen CLIP ViT-B/32 features + a temporal transformer, on a resampled long-tailed UCF-101. Pipeline in `p3.py`, results and discussion in `README.md`.

## Setup

In [ ]:
!pip -q install open-clip-torch opencv-python polars pyarrow huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Data

In [ ]:
import os, glob, zipfile
from huggingface_hub import hf_hub_download

z = hf_hub_download(repo_id="quchenyuan/UCF101-ZIP", repo_type="dataset",
                    filename="UCF-101.zip", local_dir="/content/dl")
with zipfile.ZipFile(z) as f:
    f.extractall("/content/ucf101")

vids = glob.glob("/content/ucf101/**/*.avi", recursive=True)
print(len(vids), "clips,",
      len(set(os.path.basename(os.path.dirname(v)) for v in vids)), "classes")

## Run

In [ ]:
from p3 import *

results = run("/content/ucf101")

[1/7] finding videos
      13320 clips, 101 classes
[2/7] building long-tailed split
      capped to 50 classes (MAX_CLASSES)
      imbalance factor: requested 100, realised 20.9 (tail clamped by floor/pool size)
      1913 clips kept; largest 167, smallest 8, imbalance factor 20.9
      distribution: CricketShot=167, HorseRiding=152, PlayingDhol=138, Drumming=126, BenchPress=115, Punch=104, PlayingSitar=95, Bowling=86, WritingOnBoard=79, BaseballPitch=72, Diving=65, PoleVault=59, SoccerJuggling=54, Archery=49, RockClimbingIndoor=45, Kayaking=41, TableTennisShot=37, GolfSwing=34, FrontCrawl=31, SoccerPenalty=28, Typing=25, Basketball=23, BoxingSpeedBag=21, BabyCrawling=19, BasketballDunk=18, LongJump=16, Haircut=15, WallPushups=13, YoYo=12, FieldHockeyPenalty=11, Surfing=10, HulaHoop=9, HorseRace=8, JumpingJack=8, PommelHorse=8, JugglingBalls=8, RopeClimbing=8, JavelinThrow=8, VolleyballSpiking=8, ParallelBars=8, BodyWeightSquats=8, StillRings=8, HandstandWalking=8, Rafting=8, MoppingF

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


  50/1913  12s  eta 444s
  100/1913  24s  eta 434s
  150/1913  33s  eta 387s
  200/1913  40s  eta 345s
  250/1913  55s  eta 364s
  300/1913  75s  eta 403s
  350/1913  87s  eta 390s
  400/1913  95s  eta 358s
  450/1913  102s  eta 333s
  500/1913  110s  eta 312s
  550/1913  118s  eta 293s
  600/1913  137s  eta 299s
  650/1913  154s  eta 300s
  700/1913  171s  eta 296s
  750/1913  186s  eta 288s
  800/1913  193s  eta 268s
  850/1913  201s  eta 251s
  900/1913  208s  eta 234s
  950/1913  221s  eta 224s
  1000/1913  229s  eta 209s
  1050/1913  241s  eta 198s
  1100/1913  253s  eta 187s
  1150/1913  268s  eta 178s
  1200/1913  284s  eta 169s
  1250/1913  295s  eta 156s
  1300/1913  306s  eta 144s
  1350/1913  318s  eta 133s
  1400/1913  330s  eta 121s
  1450/1913  339s  eta 108s
  1500/1913  355s  eta 98s
  1550/1913  363s  eta 85s
  1600/1913  373s  eta 73s
  1650/1913  391s  eta 62s
  1700/1913  400s  eta 50s
  1750/1913  410s  eta 38s
  1800/1913  421s  eta 26s
  1850/1913  436s  eta 15s


<string>:338: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True


      overall 0.9506  macro 0.8109  worst 0.0000
[6/7] mitigations
      class-balanced: macro 0.7855  worst 0.0000
      focal:          macro 0.7764  worst 0.0000
[7/7] polars vs pandas on the feature store
      polars            1.2 ms
      pandas naive    665.1 ms (560.7x)
      pandas fair       2.2 ms (1.8x)

saved /content/drive/MyDrive/ucf101_lt/results.json


## Results

In [ ]:
import numpy as np

print(f"{'model':16s} {'overall':>8s} {'macro':>8s} {'worst':>7s} {'head':>7s} {'tail':>7s} {'gap':>8s} {'rho':>7s}")
for k in ['zero_shot', 'ce', 'class_balanced', 'focal']:
    r = results[k]
    print(f"{k:16s} {r['overall']:8.4f} {r['macro']:8.4f} {r['worst_group']:7.4f} "
          f"{r['head_acc']:7.4f} {r['tail_acc']:7.4f} {r['head_minus_tail']:+8.4f} "
          f"{r['freq_acc_spearman']:+7.4f}")

zs, ce = results['zero_shot']['per_class'], results['ce']['per_class']
tc = results['train_counts']
order = sorted(ce, key=lambda c: -tc.get(c, 0))

print("\nclasses with zero accuracy:")
for c in order:
    if ce[c] == 0.0:
        print(f"  {c:22s} train_n={tc.get(c,0):4d}  zero-shot={zs[c]:.2f}")

third = len(order) // 3
print("\n            zero-shot   trained")
for name, grp in [("head", order[:third]), ("tail", order[-third:])]:
    print(f"  {name:6s}    {np.mean([zs[c] for c in grp]):.4f}     {np.mean([ce[c] for c in grp]):.4f}")

model             overall    macro   worst    head    tail      gap     rho
zero_shot          0.8025   0.7039  0.0000  0.7905  0.5938  +0.1967 +0.1448
ce                 0.9506   0.8109  0.0000  0.9915  0.5938  +0.3977 +0.4224
class_balanced     0.9312   0.7855  0.0000  0.9628  0.5625  +0.4003 +0.3522
focal              0.9330   0.7764  0.0000  0.9767  0.5625  +0.4142 +0.4166

classes with zero accuracy:
  YoYo                   train_n=   8  zero-shot=0.00
  PommelHorse            train_n=   6  zero-shot=0.00
  RopeClimbing           train_n=   6  zero-shot=0.00
  JavelinThrow           train_n=   6  zero-shot=0.50
  StillRings             train_n=   6  zero-shot=0.00

            zero-shot   trained
  head      0.7905     0.9915
  tail      0.5938     0.5938


## Bootstrap CI, head vs tail

In [ ]:
import numpy as np
from collections import Counter

items = find_videos("/content/ucf101")
lt, counts, classes, rif = make_long_tailed(items)
train, test = split_train_test(lt)
train_paths = set(p for p, _ in train)

feats, labels, paths = load_feature_store(
    "/content/drive/MyDrive/ucf101_lt/clip_features.parquet")
is_tr = np.array([p in train_paths for p in paths])
Xtr, Xte = feats[is_tr], feats[~is_tr]
ytr_lbl = [l for l, t in zip(labels, is_tr) if t]
yte_lbl = [l for l, t in zip(labels, is_tr) if not t]
ytr = np.array([classes.index(l) for l in ytr_lbl])
yte = np.array([classes.index(l) for l in yte_lbl])
tc = Counter(ytr_lbl)

m = TemporalTransformer(n_classes=len(classes)).fit(Xtr, ytr, epochs=30)
pred_ce = m.predict(Xte)
pred_zs, _ = zero_shot(Xte, yte_lbl, classes)

order = sorted(classes, key=lambda c: -tc.get(c, 0))
third = len(order) // 3

def stats(pred, y, idx):
    p, yy = pred[idx], y[idx]
    per = {}
    for c in classes:
        i = classes.index(c); mm = yy == i
        if mm.sum(): per[c] = (p[mm] == yy[mm]).mean()
    return (np.mean(list(per.values())),
            np.mean([per[c] for c in order[:third] if c in per]),
            np.mean([per[c] for c in order[-third:] if c in per]))

rng = np.random.RandomState(0)
B, n = 300, len(yte)
out = {'zs': [], 'ce': []}
for _ in range(B):
    idx = rng.randint(0, n, n)
    out['ce'].append(stats(pred_ce, yte, idx))
    out['zs'].append(stats(pred_zs, yte, idx))

for k in ['zs', 'ce']:
    a = np.array(out[k]); print(k)
    for j, nm in enumerate(['macro', 'head', 'tail']):
        lo, hi = np.percentile(a[:, j], [2.5, 97.5])
        print(f"   {nm:6s} {a[:,j].mean():.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

d = np.array(out['ce'])[:, 2] - np.array(out['zs'])[:, 2]
lo, hi = np.percentile(d, [2.5, 97.5])
print(f"\ntail(trained) - tail(zero-shot): {d.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")
print("CI includes 0 ->", bool(lo <= 0 <= hi))

      capped to 50 classes (MAX_CLASSES)
      imbalance factor: requested 100, realised 20.9 (tail clamped by floor/pool size)


<string>:338: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


zs
   macro  0.7083  95% CI [0.6639, 0.7608]
   head   0.7898  95% CI [0.7590, 0.8254]
   tail   0.5882  95% CI [0.4524, 0.7328]
ce
   macro  0.8194  95% CI [0.7774, 0.8667]
   head   0.9915  95% CI [0.9814, 1.0000]
   tail   0.5929  95% CI [0.4589, 0.7500]

tail(trained) - tail(zero-shot): +0.0047  95% CI [-0.2007, +0.1944]
CI includes 0 -> True
